# 00 â€” Environment and reproduction contract

This notebook establishes the executable foundation for the progressive 2017 Transformer reproduction.

| Property | Value |
| --- | --- |
| Mapped issue | [#1](https://github.com/majorgilles/transformer-2017-reproduction/issues/1) |
| Depends on | None; first notebook |
| Canonical host | Native Windows with an RTX 4070 SUPER |
| CI host | Linux CPU for portability and freshness, not the canonical CUDA claim |

The notebook deliberately contains no data, tokenizer, embedding, attention, or model implementation.


In [1]:
#| default_exp environment


## Frozen environment decisions

| Decision | Contract |
| --- | --- |
| Python | CPython 3.12, pinned by `.python-version` and `uv.lock` |
| Resolver | uv 0.12.x with a committed lockfile |
| Windows PyTorch | PyTorch 2.11 from the explicit CUDA 12.8 wheel index |
| Linux CI PyTorch | PyTorch 2.11 from the explicit CPU wheel index |
| Literate source | Jupyter notebooks exported by nbdev 3.x |
| Formatting/linting | Ruff |
| Static typing | Pyright strict mode for exported project code |
| Tests | nbdev notebook tests with plain-assertion focused checks |

PyTorch's bundled CUDA runtime does not require a separate system CUDA toolkit for this project. The NVIDIA driver must be compatible, and diagnostics must prove that the locked wheel can see the actual GPU.


## Typed diagnostic API

The exported API returns a bounded, JSON-safe snapshot. It intentionally avoids machine-specific absolute paths and environment variables so notebook output can be committed safely.


In [2]:
#| export
from __future__ import annotations

import argparse
import hashlib
import importlib.metadata as metadata
import json
import platform
import subprocess
import sys
from collections.abc import Sequence
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Final, Protocol, cast

import torch


def _find_project_root(start: Path) -> Path:
    candidate = start.resolve()
    if candidate.is_file():
        candidate = candidate.parent
    for directory in (candidate, *candidate.parents):
        if (directory / "uv.lock").is_file():
            return directory
    raise FileNotFoundError(f"could not find uv.lock from {start}")


PROJECT_ROOT: Final[Path] = _find_project_root(
    Path(__file__) if "__file__" in globals() else Path.cwd()
)


class _CudaDeviceProperties(Protocol):
    name: str
    total_memory: int


class _CudaApi(Protocol):
    def is_available(self) -> bool: ...

    def get_device_properties(self, device: int) -> _CudaDeviceProperties: ...


@dataclass(frozen=True, slots=True)
class EnvironmentReport:
    os: str
    os_release: str
    architecture: str
    python: str
    torch: str
    torch_cuda_runtime: str | None
    cuda_available: bool
    gpu_name: str | None
    gpu_memory_mib: int | None
    nvidia_driver: str | None
    uv: str | None
    nbdev: str
    ruff: str
    pyright: str
    lock_sha256: str


def _version(distribution: str) -> str:
    return metadata.version(distribution)


def _command_version(command: Sequence[str]) -> str | None:
    try:
        completed = subprocess.run(
            command,
            check=True,
            capture_output=True,
            text=True,
            timeout=10,
        )
    except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
        return None
    first_line = completed.stdout.strip().splitlines()
    return first_line[0] if first_line else None


def _lock_sha256(project_root: Path) -> str:
    lock_path = project_root / "uv.lock"
    return hashlib.sha256(lock_path.read_bytes()).hexdigest()


def collect_environment_report(project_root: Path | None = None) -> EnvironmentReport:
    root = (project_root or PROJECT_ROOT).resolve()
    cuda = cast(_CudaApi, torch.cuda)
    cuda_available = cuda.is_available()
    gpu_name: str | None = None
    gpu_memory_mib: int | None = None
    if cuda_available:
        properties = cuda.get_device_properties(0)
        gpu_name = properties.name
        gpu_memory_mib = round(properties.total_memory / 1024**2)

    driver_line = _command_version(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]
    )
    return EnvironmentReport(
        os=platform.system(),
        os_release=platform.release(),
        architecture=platform.machine(),
        python=platform.python_version(),
        torch=torch.__version__,
        torch_cuda_runtime=torch.version.cuda,
        cuda_available=cuda_available,
        gpu_name=gpu_name,
        gpu_memory_mib=gpu_memory_mib,
        nvidia_driver=driver_line,
        uv=_command_version(["uv", "--version"]),
        nbdev=_version("nbdev"),
        ruff=_version("ruff"),
        pyright=_version("pyright"),
        lock_sha256=_lock_sha256(root),
    )


def report_as_json(report: EnvironmentReport) -> str:
    return json.dumps(asdict(report), indent=2, sort_keys=True)


def validate_canonical_windows_cuda(report: EnvironmentReport) -> tuple[str, ...]:
    problems: list[str] = []
    if report.os != "Windows":
        problems.append("canonical diagnostics require native Windows")
    if not report.cuda_available:
        problems.append("the locked PyTorch build cannot access CUDA")
    if report.gpu_name != "NVIDIA GeForce RTX 4070 SUPER":
        problems.append(f"expected RTX 4070 SUPER, found {report.gpu_name!r}")
    if report.gpu_memory_mib is None or report.gpu_memory_mib < 12_000:
        problems.append("expected at least 12,000 MiB of reported GPU memory")
    return tuple(problems)


def main(argv: Sequence[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Report the locked Transformer environment")
    parser.add_argument("--json", action="store_true", help="emit machine-readable JSON")
    parser.add_argument(
        "--require-canonical-gpu",
        action="store_true",
        help="fail unless native Windows can access the expected RTX 4070 SUPER",
    )
    args = parser.parse_args(argv)
    report = collect_environment_report()
    print(report_as_json(report) if args.json else report)
    if args.require_canonical_gpu:
        problems = validate_canonical_windows_cuda(report)
        if problems:
            print("; ".join(problems), file=sys.stderr)
            return 1
    return 0

## Focused contract tests

These tests exercise the JSON boundary and canonical-host validation without fabricating a CUDA device.


In [3]:
from dataclasses import replace

report = collect_environment_report()
payload = json.loads(report_as_json(report))
assert payload["python"].startswith("3.12.")
assert payload["lock_sha256"] == _lock_sha256(PROJECT_ROOT)
assert len(payload["lock_sha256"]) == 64

noncanonical = replace(report, os="Linux", cuda_available=False, gpu_name=None, gpu_memory_mib=None)
problems = validate_canonical_windows_cuda(noncanonical)
assert "canonical diagnostics require native Windows" in problems
assert "the locked PyTorch build cannot access CUDA" in problems

## Visible native-Windows evidence

The following cell captures the bounded report from the locked environment. Acceptance requires `cuda_available=true`, the RTX 4070 SUPER name, at least 12,000 MiB reported VRAM, and an empty canonical validation result.


In [4]:
report = collect_environment_report()
print(report_as_json(report))
assert validate_canonical_windows_cuda(report) == ()

{
  "architecture": "AMD64",
  "cuda_available": true,
  "gpu_memory_mib": 12282,
  "gpu_name": "NVIDIA GeForce RTX 4070 SUPER",
  "lock_sha256": "d8aa12021d0571944348e1f03b7acdd9cc9359394b16356b78b031254b6f134a",
  "nbdev": "3.3.13",
  "nvidia_driver": "616.56",
  "os": "Windows",
  "os_release": "11",
  "pyright": "1.1.411",
  "python": "3.12.10",
  "ruff": "0.16.5",
  "torch": "2.11.0+cu128",
  "torch_cuda_runtime": "12.8",
  "uv": "uv 0.12.0 (b88d7c5c4 2026-07-28 x86_64-pc-windows-msvc)"
}


## Quality matrix

Run the full native-Windows gate with:

```powershell
powershell -ExecutionPolicy Bypass -File scripts/quality.ps1
```

CI repeats the portable subset on Linux CPU. The export freshness step runs `nbdev-export` and then requires a clean Git diff.

| Check | Command |
| --- | --- |
| Locked installation | `uv sync --locked` |
| Formatting | `uv run ruff format --check .` |
| Lint | `uv run ruff check .` |
| Strict typing | `uv run pyright` |
| Export freshness | `uv run nbdev-export` then `git diff --exit-code` |
| Notebook test | `uv run nbdev-test --path notebooks/00_environment_contract.ipynb` |
| Canonical GPU | `uv run transformer-env --json --require-canonical-gpu` |

The lockfile SHA-256 in the visible report identifies the exact resolved environment used for approval.


## Explicitly deferred

Data schemas, fixtures, WMT rights work, tokenization, embeddings, attention, and all model/training behavior belong to later notebooks. This notebook exports environment diagnostics only.


## HITL checkpoint

Before issue #1 is approved, the maintainer reviews:

1. the captured RTX 4070 SUPER report;
2. Python 3.12, PyTorch 2.11/CUDA 12.8, uv 0.12, nbdev 3.x, Ruff, and Pyright versions;
3. the passing quality matrix; and
4. the fact that no later notebook scope was implemented.
